In [ ]:
import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler
from pathlib import Path 

In [ ]:
TRAIN = "BFIPPDSR_project/data/raw/nsl/KDDTrain+.csv"
TEST  = "BFIPPDSR_project/data/raw/nsl/KDDTest+.csv"

cols = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes','land','wrong_fragment',
    'urgent','hot','num_failed_logins','logged_in','num_compromised','root_shell','su_attempted',
    'num_root','num_file_creations','num_shells','num_access_files','num_outbound_cmds',
    'is_host_login','is_guest_login','count','srv_count','serror_rate','srv_serror_rate',
    'rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate','srv_diff_host_rate',
    'dst_host_count','dst_host_srv_count','dst_host_same_srv_rate','dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate','dst_host_serror_rate',
    'dst_host_srv_serror_rate','dst_host_rerror_rate','dst_host_srv_rerror_rate',
    'label','difficulty'
]

# Load
df_train = pd.read_csv(TRAIN, names=cols)
df_test  = pd.read_csv(TEST,  names=cols)


In [ ]:
import os

print(os.path.exists("BFIPPDSR_project/data/raw/nsl/KDDTrain+.csv"))
print(os.path.exists("BFIPPDSR_project/data/raw/nsl/KDDTest+.csv"))
print(os.getcwd())

True
True
c:\Users\88019\Desktop\BFIPPDSR_project\src


In [21]:
print(df_train.columns)
print(df_test.columns)

Index(['duration', 'protocol_type', 'service', 'flag', 'src_bytes',
       'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
       'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
       'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
       'num_access_files', 'num_outbound_cmds', 'is_host_login',
       'is_guest_login', 'count', 'srv_count', 'serror_rate',
       'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
       'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
       'dst_host_srv_count', 'dst_host_same_srv_rate',
       'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
       'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
       'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
       'dst_host_srv_rerror_rate'],
      dtype='object')
Index(['duration', 'protocol_type', 'service', 'flag', 'src_bytes',
       'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
       'num_failed_logins',

In [23]:
df_train = pd.read_csv(TRAIN, names=cols)
df_test  = pd.read_csv(TEST,  names=cols)

# Normalize labels
df_train['label'] = df_train['label'].str.strip().str.lower()
df_test['label']  = df_test['label'].str.strip().str.lower()

# Binary encoding
y_train = (df_train['label'] != 'normal').astype(int).values
y_test  = (df_test['label']  != 'normal').astype(int).values

print("NSL train labels unique:", np.unique(y_train))
print("NSL test labels unique:", np.unique(y_test))

# Drop label + difficulty before encoding
df_train = df_train.drop(columns=['label','difficulty'])
df_test  = df_test.drop(columns=['label','difficulty'])

NSL train labels unique: [0 1]
NSL test labels unique: [0 1]


In [24]:
# One-hot encode categorical features
cat = ['protocol_type','service','flag']
df_train = pd.get_dummies(df_train, columns=cat)
df_test  = pd.get_dummies(df_test,  columns=cat)

# Align features (label is already gone!)
X_train, X_test = df_train.align(df_test, join='outer', axis=1, fill_value=0)

In [25]:
print(X_train.dtypes)
print(X_train.head())


count                       int64
diff_srv_rate             float64
dst_bytes                   int64
dst_host_count              int64
dst_host_diff_srv_rate    float64
                           ...   
srv_rerror_rate           float64
srv_serror_rate           float64
su_attempted                int64
urgent                      int64
wrong_fragment              int64
Length: 122, dtype: object
   count  diff_srv_rate  dst_bytes  dst_host_count  dst_host_diff_srv_rate  \
0      2           0.00          0             150                    0.03   
1     13           0.15          0             255                    0.60   
2    123           0.07          0             255                    0.05   
3      5           0.00       8153              30                    0.00   
4     30           0.00        420             255                    0.00   

   dst_host_rerror_rate  dst_host_same_src_port_rate  dst_host_same_srv_rate  \
0                  0.05                         0.

In [26]:
print([c for c in X_train.columns if X_train[c].dtype == 'object'])


[]


In [27]:
# Convert problematic columns to numeric
for col in ['dst_host_srv_rerror_rate', 'duration']:
    X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
    X_test[col]  = pd.to_numeric(X_test[col], errors='coerce')

# Now convert everything to float32
X_train = X_train.to_numpy(dtype=np.float32)
X_test  = X_test.to_numpy(dtype=np.float32)

# Scale
from sklearn.preprocessing import MaxAbsScaler

scaler = MaxAbsScaler()
X_train = scaler.fit_transform(X_train.astype(np.float32))
X_test  = scaler.transform(X_test.astype(np.float32))


In [ ]:
np.save("BFIPPDSR_project/data/processed/nsl_X_train.npy", X_train)
np.save("BFIPPDSR_project/data/processed/nsl_y_train.npy", y_train)
np.save("BFIPPDSR_project/data/processed/nsl_X_test.npy",  X_test)
np.save("BFIPPDSR_project/data/processed/nsl_y_test.npy",  y_test)
print("NSL-KDD processed and saved.")

NSL-KDD processed and saved.
